# VLM methodology diagnostics

This notebook records reproducible evidence for substantial dataset or scientific-method changes. Exact source changes belong in Git, release-facing summaries belong in `CHANGELOG.md`, and scientific decisions belong in `docs/development_changes.md`.

The initial entry establishes the input inventory for the unchanged module at commit `1c4ab96fc29d9f1d174e99ed763a7eca02a72abc`. Local data and generated outputs remain under the ignored `data/` directory.

In [ ]:
from pathlib import Path
import hashlib

import xarray as xr

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
reference_dir = repo_root / "data" / "inputs" / "reference"
reference_dir

In [ ]:
def md5(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


inventory = []
for path in sorted(reference_dir.iterdir()):
    if not path.is_file():
        continue
    with xr.open_dataset(path) as dataset:
        inventory.append(
            {
                "file": path.name,
                "bytes": path.stat().st_size,
                "md5": md5(path),
                "dimensions": dict(dataset.sizes),
                "coordinates": sorted(dataset.coords),
                "data_variables": sorted(dataset.data_vars),
            }
        )
inventory

## Fixed-seed baseline check

The initial one-site baseline was run twice in the development container with 10 samples, base year 2000, projection years 2000--2100, and seed 1234. Scientific contents must match even though the global `history` timestamp makes the raw NetCDF file checksums differ.

In [ ]:
reference_output = repo_root / "data" / "outputs" / "reference" / "lslr.nc"
repeat_output = repo_root / "data" / "outputs" / "reference" / "lslr_repeat.nc"

with (
    xr.open_dataset(reference_output) as first,
    xr.open_dataset(repeat_output) as repeat,
):
    xr.testing.assert_equal(first, repeat)
    baseline_summary = {
        "scientific_contents_equal": True,
        "sizes": dict(first.sizes),
        "dimensions": list(first["sea_level_change"].dims),
        "dtype": str(first["sea_level_change"].dtype),
        "units": first["sea_level_change"].attrs["units"],
        "minimum_mm": float(first["sea_level_change"].min()),
        "maximum_mm": float(first["sea_level_change"].max()),
        "history_timestamps_differ": first.attrs["history"] != repeat.attrs["history"],
    }
baseline_summary

## Candidate comparison template

For each methodological change, record the hypothesis, affected inputs/functions, and expected differences here. Run reference and candidate configurations with the same locations, projection years, sample count, chunk size, and seed. Compare intermediate fields as well as the final NetCDF dimensions, metadata, missing values, quantiles, and site-level differences. Keep large tables and figures under `data/diagnostics/`.

## 2026-11-02 candidate correction: evidence and assumptions

Two executed exploratory notebooks and decadal baseline/candidate mean/std files are kept locally under `data/discovery/vlm_review_20261102/` and ignored by Git. See that workspace's README for exact commands, parameters, source snapshots, and input SHA256 hashes.

The same five input files, 500 samples, seed 1234, base year 2000, 2000–2150 every ten years, six Figure 4 cities, and 14,366 coastal nodes were used in both runs. Coastal-node RMS differences at 2150 are 66.60 mm (mean) and 50.88 mm (standard deviation). The main scientific change integrates the velocity taper, retaining historical non-GIA displacement rather than erasing it in 2150. No CMR reanalysis or input replacement was made.

Scientific and schema regression tests live in `tests/test_science.py` and `tests/test_integration.py`. The following small check remains reproducible without the published archive. GIA sea-surface covariance, spatial error covariance, and actual GRW sample paths remain limitations; see `docs/development_changes.md`.


In [ ]:
import sys
from pathlib import Path
import numpy as np

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root / "src"))
from oelsmann24_verticallandmotion.vlm_postprocess import integrated_decay  # noqa: E402

years = np.array([2000, 2050, 2100, 2150, 2300])
displacement = 2 * integrated_decay(years, 2000)  # 2 mm/year non-GIA subsidence
np.testing.assert_allclose(displacement[-2:], [200, 200])
np.testing.assert_allclose(
    integrated_decay(years, 2100),
    integrated_decay(years, 2000) - integrated_decay([2100], 2000),
)
dict(zip(years.tolist(), displacement.tolist()))